# Introduction to Neuroinformatics
## Exercise session 4: Active membrane properties

Welcome to the fourth exercise of the introduction to neuroinformatics course. In the previous exercise, we introduced the **Integrate-and-No-Fire** neuron model, focusing on how synaptic inputs allow currents to alter the membrane potential of a neuron. We explored how these <ins>inputs are integrated and summed at the soma</ins>, determining the overall effect on the neuron's membrane potential. Additionally, we briefly touched on the concept of the **action potential**, a <ins>non-linear, all-or-nothing response</ins> triggered when the <ins>membrane potential crosses a specific threshold</ins>. This **"spike"** represents a critical event in neural communication, as it enables the neuron to send signals to downstream targets.

In the following exercise we will see that different types of inputs can have different effects on the postsynaptic membrane potential. Further, we will introduce the Integrate-and-Fire neuron, a model that can produce **spikes**.

This notebook is designed to be run on Google Colab. If you wish to save your progress, click on the "Copy to Drive" button.

---

# Table of contents

* [Packages](#packages)
* [1: Synapses](#1)
  * [1.1: Synaptic input](#1.1)
  * [1.2: Equivalent circuit of a synapse](#1.2)
* [2: Input integration and the Integrate-and-Fire model](#2)
  * [2.1: Temporal EPSP integration](#2.1)
  * [2.2: The Hodgkin-Huxley model](#2.2)
  * [2.3: The Integrate-and-Fire neuron model](#2.3)
  * [2.4: The Leaky-Integrate-and-Fire Neuron model](#2.4)
* [3: Further Readings](#further)

---

# Packages <a name="packages"></a>

The following cells install the necessary packages and download the files for the exercise.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"IN_COLAB: {IN_COLAB}")

In [ ]:
if IN_COLAB:
    # Install necessary packages.
    !pip3 install --quiet ipywidgets

In [ ]:
if IN_COLAB:
    !mkdir -p utils_ex4
    !mkdir -p utils_ex4/figures

    # -O rather than -P for the code and figures: wget -P leaves an existing file
    # alone and writes name.1 beside it, so a corrected helper would never reach a
    # runtime that has run this cell before. This notebook fetches no bulk data.
    !wget -O utils_ex4/utils.py https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex4/utils.py
    !wget -O utils_ex4/figures/ex4_synapse_circuit.png https://github.com/ManteLab/Iton_notebooks_public/raw/refs/heads/main/utils_ex4/figures/ex4_synapse_circuit.png

---

# 1: Synapses <a name="1"></a>

---

## 1.1: Synaptic input <a name="1.1"></a>



Recall the following **key concepts**:

**Depolarization**: Neurons <ins>at rest</ins> have a <ins>membrane potential</ins> of approximately <ins>-70 mV</ins>, which we refer to as a <ins>polarized</ins> state (i.e., a state far from 0 mV). **Depolarizing currents** bring the membrane potential <ins>closer to 0 mV</ins>.

**Hyperpolarization**: In some cases, synaptic input results in a **hyperpolarizing effect**, driving the membrane potential even further away from 0 mV, thus making the neuron less likely to fire.

**EPSP** (**E**xcitatory **P**ost**s**ynaptic **P**otential): Synaptic input that has a depolarizing effect, moving the membrane potential closer to the threshold for an action potential.

**IPSP** (**I**nhibitory **P**ost**s**ynaptic **P**otential): Synaptic input that causes hyperpolarization, moving the membrane potential further from the firing threshold.

Whether a <ins>synaptic input is excitatory or inhibitory</ins> depends on the **neurotransmitter** (**NT**) released by the presynaptic neuron. According to **Dale's Principle**, a neuron releases the <ins>same neurotransmitter at all of its synapses throughout its lifetime</ins>, maintaining a consistent chemical action.

In the brain, the <ins>primary excitatory neurotransmitter</ins> is **glutamate**, while the <ins>main inhibitory neurotransmitter</ins> is **GABA** (gamma-Aminobutyric acid).

Note that so far all synaptic inputs in our models have been excitatory.

---

## 1.2: Equivalent circuit of a synapse <a name="1.2"></a>

The figure below depicts a simplified neuron, represented by an equivalent circuit including a fast chemical synapse. $V_{rest}$ is the resting potential of the cell (here denoted as the **reversal potential** of the **leakage current** $E_L$), $R_L$ is the input resistance and $V_m(t)$ is the membrane potential dependent on time $t$.

---

Note, that <ins>synaptic input</ins> occurs only if <ins>specific ion channels</ins> in the postsynaptic membrane <ins>open</ins>. These channels are often <ins>selectively permeable for multiple ion types</ins> and in order for them to open, they have to bind <ins>presynaptically released NT</ins>.

---

In the example below, $E_{syn}$ is the equilibrium potential of the synapse and $g_{syn}$ is the synaptic conductance. Following Ohm's law you obtain the synaptic current $I_{syn}$ (generated by a single synapse depicted on the left side of the diagram by:

$I_{syn} = g_{syn}(t) (V_m(t) - E_{syn})$


Note that the synaptic conductance $g_{syn}$ is a function of time and depends on the opening of synaptic ion channels. The complete equation for the above circuit (also implementing Kirchhoff's law) yields:

$0 = I_{syn} + I_L = g_{syn}(t) (V_m(t) - E_{syn}) + (V_m(t) - V_{rest}) / R_L$


Assume that the synapse is "switched on" at t = 0 (after a presynaptic action
potential arrives at the synapse and NT binds to the postsynaptic channels), and current may pass through. Further, $g_{syn}$ is "switched off" again at t = 1 (when all NT has been cleared again from the synaptic cleft):

$g_{syn}(t) =
\begin{cases}
0 & \text{for } t < 0, \\
1 nS & \text{for } 0 < t \leq 1, \\
0 & \text{for } t > 1
\end{cases}$

Assume $V_{rest} = -70\text{mV}$ and $R_L = 2G\Omega$.

In [ ]:
# Colab's markdown renderer cannot resolve relative paths to files on the VM,
# so the figure is displayed from Python instead.
from IPython.display import Image, display

display(Image("utils_ex4/figures/ex4_synapse_circuit.png", width=900))


---

> **Assignment 1**
>
> Let $E_{syn} = 10\text{mV}$. Is this an excitatory or inhibitory synapse? Calculate $I_{syn}$ and the amplitude of the postsynaptic potential (i.e. $V_m(t)$ for $0\text{ms} < t < 1\text{ms}$).

**Solution:**



> **Assignment 2**
>
> Let $E_{syn} = -90\text{mV}$. Is this an excitatory or inhibitory synapse? Calculate $I_{syn}$ and the amplitude of the postsynaptic potential.


**Solution:**



> **Assignment 3**
>
> Let $E_{syn} = -70\text{mV}$. Calculate $I_{syn}$ and the amplitude of the postsynaptic potential. Calculate $g_{total}$ (total membrane conductance). What effect will this synapse have on the responsiveness of the membrane? Is this an excitatory or inhibitory synapse?

**Solution:**

---


---

# 2: Input integration and the Integrate-and-Fire model <a name="2"></a>



---

## 2.1: Temporal EPSP integration <a name="2.1"></a>

In [ ]:
from utils_ex4.utils import iplot_InoF_model
# Integrate and No fire
iplot_InoF_model()

The **interactive plot** above allows you to simulate input to a neuron with a single synapse.  *Note that the new design of the interactive plot requires you to click "**Update Plot**", after you changed parameters. This should allow the code in the background to run faster.*

Depending on how many action potentials the presynaptic cell fires (Input Frequency), more or fewer EPSPs accumulate at the soma. Change the parameters to get and feeling for their effect on the membrane potential in this model and attend to the **assignments** below.


---

> **Assignment 4**
>
> Change the parameter "Distance: > Synapse-Soma" while keeping other parameters unchanged. Then, change the "Input Frequency" while keeping other parameters unchanged. Explain why these changes introduce different delays in the occurrence of EPSPs.

**Solution:**



> **Assignment 5**
>
> Does the model incorporate the membrane time constant $\tau$ or the  length constant $\lambda$? Explain how you came to your conclusion.

**Solution:**

---


---

## 2.2: The Hodgkin-Huxley model <a name="2.2"></a>

The **model presented in Section 2.a** allows for indefinite depolarization of a neuron by linearly summing all EPSPs arriving at the soma. However, as we have seen, this behavior does not at all represent how real neurons function. When the membrane potential crosses a certain **depolarization threshold**, neurons generate a <ins>non-linear, all-or-nothing response</ins>: **The action potential**.

In 1952, Alan Hodgkin and Andrew Huxley developed a <ins>mathematical model</ins> that explains how action potentials are <ins>initiated and propagated</ins> along the squid giant axon.

$I_{\text{ion}} =
\underbrace{g_{L} (V_{m} - E_{L})}_{\text{Leak current, }I_{L}} +
\underbrace{\overline{g_{K}} *  n^4 (V{m} - E_{K})}_{\text{Potassium current, }I_{K}} +
\underbrace{\overline{g_{Na}} * m^3 * h (V_{m} - E_{Na})}_{\text{Sodium current, }I_{Na}}
$

The **total ionic current** $I_{ion}$ is a <ins>sum</ins> of the <ins>constant currents</ins> through so called **leak channels**, $I_{L}$ and the <ins>currents that are responsible for fast de- and repolarisation during an action potential</ins>: $I_{Na}$ and $I_{K}$. As we have seen in previous exercises, the current for each ion depends on **a)** the selective permeability of the membrane for the ion species (conductance, $g_{ion}$), and **b)** the difference between the membrane potential and the reversal potential of the ion species:

$I_{ion} = g_{ion} * (Vm - E_{ion})$

For several ions, the conductance $g_{ion}$ is not constant but modulated by a physical gate that is opened or closed with a certain probability. The probability for one of the ion channels to be open is given by $n_{ion}$. Thus, $n_{ion}$ can be interpreted as a proportion of the maximum conductance.

The actual ionic conductance can therefore be written as the maximal conductance $\overline{g_{ion}}$ multiplied by the gating variable so that:

$g_{ion}$= $\overline{g_{ion}}$  * $n_{ion}$

With this is mind, attend to the following **assignments**:


---

> **Assignment 6**
>
> Answer the following questions when interpreting the Hodgkin-Huxley model:
>
> 1. What are the lower and upper bound for $n_{ion}$?
> 2. Suppose that each channel has several gates in series (let's say $x$ gates), all of which have to be open to permit ions to pass through the channel. If the probability of a single gate to be open is $n$, what is the probability that all $x$ gates are open?
> 3. Each gate can change from an open to a closed state and vice-versa. If the probability that a gate in a closed state will open is $\alpha_n$, and the probability that an open gate closes is $\beta_n$, how does the proportion of open gates $n$ change? $dn / dt =$ . . .
> 4. In steady state, $n$ does not change anymore: $dn / dt = 0$. What is the gating variable $n$ in terms of $\alpha_n$ and $\beta_n$ at steady? $n_{∞} =$ . . .
> 5. Hodgkin & Huxley found that a model with 4 gates in series produces a good fit to the S-shaped curve of the potassium current ($I_{K}$) during the action potential. Explain the second term ($I_{K}$) of their model.
> 6. In the third term ($I_{Na}$) Hodgkin & Huxley introduced another variable termed $h$. It reflects a specific property of the sodium channels involved in action potential generation: Inactivation. Once, these sodium channels have opened and closed again, they will resist to open again for a certain amount of time. Explain the third term ($I_{Na}$) of their model.

**Solution:**

---


You have already watched this model run. In **notebook 1, Section 1.2** (*The action potential*), the cell `iplot_action_potential()` integrates exactly these equations: it plots the membrane potential in black together with the three gating variables $m$, $h$ and $n$ on the right-hand axis, with a slider for the injected current.

That figure is worth returning to now that the equations are in front of you. Watch which gate moves first once the potential crosses threshold, which one shuts the sodium current down again, and what all three are doing while the potential overshoots and then recovers. Does the shape of that trace, and the theory behind it, make more sense to you now than it did in the first session?

---

## 2.3: The Integrate-and-Fire neuron model <a name="2.3"></a>

In <ins>contrast to the Integrate-and-no-Fire model</ins>, the **Integrate-and-Fire neuron model** triggers a **spike** (action potential) as soon as threshold is reached and resets the membrane potential to resting potential immediately afterwards.

In the **interactive plot** below we extend the previous model (section 2.a) to a spiking (IF neuron) model. The second plot below is identical to the first plot in section 2.a, except that we now added an adjustable parameter: the action potential threshold. The first plot shows the resulting spike train resulting from the depolarization levels in plot 2.

Feel free to change parameters of the model and see what happens.

In [ ]:
from utils_ex4.utils import iplot_Integrate_and_Fire_model

In [ ]:
# Integrate and Fire model
iplot_Integrate_and_Fire_model()

In the **interactive plot** above, set the <ins>weight to maximum (100)</ins>, the <ins>distance to its minimum value (0.1mm)</ins>, the <ins>AP threshold to 20mV</ins> and the <ins>refractory scale factor to 0.5</ins>. Compare the spike trains between <ins>input frequencies of 220-230</ins> to input frequencies of <ins>320-330</ins>. Do another comparison with the <ins>refractory scale factor to 0.2</ins>.




---

> **Assignment 7**
>
> How do the resulting spike trains differ? What is the function of the refractory scale factor?

**Solution:**



> **Assignment 8**
>
> When looking at the resulting plot, try to relate its dynamics to the mathematical model of Hodgkin and Huxley from section 2.b. Link the terms in their formula to observations in the interactive model.

**Solution:**

---


---

## 2.4: The Leaky-Integrate-and-Fire Neuron model <a name="2.4"></a>

Slowly but surely our neuron model emulates more and more properties of a real neuron.

We have seen that, given a depolarizing current pulse at time $t$ and location $x$, the resulting EPSP attenuates over time and while propagating along a dendrite (**cable equation)**. This is due to a subset of **potassium channels** that are considered "leaky". These channels have a rather high probability of being open and are <ins>different to the repolarizing voltage-gated potassium channels</ins> found in axons involved in action potential generation. You have seen leak current ($I_{L}$) already as a part of the equivalent circuit of a synapse in section **1.b** of this exercise.

The model in the **interactive plot** below is a **Leaky-Integrate-and-Fire (LIF)** neuron - synaptic inputs now generate EPSPs that attenuate over time. Go ahead, change some of the parameters to get a feeling for the model and attend to the **assignments** below.

In [ ]:
from utils_ex4.utils import iplot_Leaky_Integrate_and_Fire_model

In [ ]:
# Leaky Integrate and Fire
iplot_Leaky_Integrate_and_Fire_model()

Since you now got a feeling on how the model behaves, try to find settings to generate one EPSP with an amplitude approximately 60% of the amplitude necessary to reach the spiking threshold. Then, independently change the distance and the time constant parameters.


---

> **Assignment 9**
>
> Does this model implement $\tau$? If so, how can you tell?

**Solution:**

---



Next, change the parameters to produce two or three EPSPs but keep the time constant at its minimum value (1ms). Then slowly increase the time constant.



---

> **Assignment 10**
>
> How does this influence the neuron’s ability to integrate information?

**Solution:**

---



Next, set the following parameters: $\text{Input frequency} = 10\text{Hz}$, $\text{weight} = 50\text{mV}$, $\text{distance} = 1\text{mm}$, $\text{threshold} = 100\text{mV}$ and $\text{time constant} = 50\text{ms}$.



---

> **Assignment 11**
>
> Calculate the membrane potential at $t=60\text{ms}$, then change the input frequency to 20\text{Hz} and calculate the membrane potential at $t=60\text{ms}$ again.

**Solution:**

---



At last, set the following parameters: $\text{Input frequency} = 10\text{Hz}$, $\text{weight} = 30\text{mV}$, $\text{distance to soma} = 0.1\text{mm}$, $\text{threshold} = 20\text{mV}$, $\text{time constant} = 30\text{ms}$ and $\text{refractory scale factor} = 0.01$.



---

> **Assignment 12**
>
> Increase the distance to soma to 1mm and then to 2mm. What changes? When comparing your observations with the results you expect from cable theory, which parameter is not implemented in this model? How would the results change, if this parameter was implemented?  


**Solution:**

---


Up to this point, all of our plots in this exercise have modeled a neuron receiving input from only a single synapse. However, in biological brains and more complex artificial neural networks, individual neurons typically receive inputs from dozens, hundreds, or even over a thousand synapses.

The interactive plot below allows you to model a neuron with **20 synapses**. In <ins>previous examples</ins>, you were able to <ins>modify parameters for a single synapse</ins>, such as its distance from the soma or its synaptic strength (weight). As you can imagine, manually setting these parameters for 20 synapses would be a time-consuming task. To simplify this, the <ins>sliders below</ins> allow you to <ins>specify **distributions**</ins> (mean and standard deviation) for these values. From these distributions, 20 random values are drawn and assigned to the 20 synapses.

In [ ]:

from utils_ex4.utils import iplot_Leaky_Integrate_and_Fire_with_distributions

In [ ]:
iplot_Leaky_Integrate_and_Fire_with_distributions()

In order to understand the power of signal integration in single neurons, we will briefly introduce the concept of coding (precisely rate and temporal coding) here. You will hear more about these concepts during the lecture, so for now it is sufficient to keep it simple.

It is in general a difficult task to assign meaning to the spiking dynamics of neurons in the brain. What does the neuron encode when it fires so and so often or at these specific times? Two common coding strategies are well described and here we introduce them briefly:

**Rate coding**: The spiking frequency of a neuron is informative about some property of the signals. For example, a sensory neuron will fire action potentials at higher frequencies, when the input stimulus is stronger.

**Temporal coding**: The firing frequency itself is not informative about the encoded information but rather temporal factors like the relative timing of single spikes.

With this in mind, set the following parameters for the neuron model above: $\text{Distance} = 1\text{mm} \pm 0.01$, $\text{Weight} = 5\text{mV} \pm 0.01$, $ \text{Time constant} = 10\text{ms}$ and $\text{Threshold} = 0\text{mV}$. Then find a set of parameters that maximises the amount of inputs while the neuron fires as few spikes as possible.






---

> **Assignment 13**
>
> Which parameters did you find? Explain their influence.

**Solution:**

---



Now change the parameters to provoke some more action potentials (2-3 spikes are sufficient) in the neuron. Individually change the parameter distributions and observe the neuron response in all your chosen parameter regimes.


---

> **Assignment 14**
>
> Does any of the distributions influence the neuron's output more than the others? Is there a particular distribution that critically influences the neuron's response so that - if not above / below a certain threshold - the neuron won't fire?

**Solution:**



> **Assignment 15**
>
> Find parameter regimes so that the neuron implements rate coding.

**Solution:**



> **Assignment 16**
>
> Find parameter regimes so that the neuron implements temporal coding.



**Solution:**

---


Set the following parameters: $\text{Weight} = 2.5\text{mV} \pm 2.5$, $ \text{Time constant} = 10\text{ms}$ and $\text{Threshold} = -20\text{mV}$.




---

> **Assignment 17**
>
> For different parameter regimes (e.g. high input rate with poor synchronization), how many inputs are required to provoke an action potential at $t=20\text{ms}$?


**Solution:**

---


---

# 3. Further Readings <a name="further"></a>

There is no assigned reading for this session — the lecture covers the material you need. The references below are for going further.

- Hodgkin, A. L. & Huxley, A. F. (1952), "A quantitative description of membrane current and its application to conduction and excitation in nerve," *Journal of Physiology* — the original paper behind Section 2.2, and still the clearest statement of what the gating variables are doing.
- Dayan, P. & Abbott, L. F. (2001), *Theoretical Neuroscience*, Chapter 5 (Model Neurons I) — synaptic input, the integrate-and-fire neuron and its leaky variant, covering Sections 1 and 2.3-2.4.
- Gerstner, W., Kistler, W., Naud, R. & Paninski, L. (2014), *Neuronal Dynamics*, Chapter 2 (Ion Channels and the Hodgkin-Huxley Model), [free online](https://neuronaldynamics.epfl.ch/online/Ch2.html) — works through the Hodgkin-Huxley equations step by step; its [Chapter 1](https://neuronaldynamics.epfl.ch/online/Ch1.html) does the same for the leaky integrate-and-fire neuron.

---

# End of this exercise session